<a href="https://colab.research.google.com/github/AngelTroncoso/Feature-Engineering/blob/main/actividad2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Técnicas de Target Encoding

**Objetivo:** Demostrar el uso de codificación basada en el objetivo (target) mediante la herramienta **Category Encoders** en Python. Esta técnica es crucial para manejar variables con alta cardinalidad sin aumentar masivamente la dimensionalidad del conjunto de datos.

## Introducción

¡Hola! Hoy vamos a dominar una de las herramientas más potentes para manejar datos complejos. A veces las categorías nos complican la vida, pero el **Target Encoding** nos permite transformar esa confusión en información pura y útil para nuestros modelos de machine learning.

### El problema de la Alta Cardinalidad

Pensemos por un momento en el problema de la **alta cardinalidad**. Cuando tenemos una columna como *Código Postal* con miles de valores distintos, usar técnicas tradicionales como el *One Hot Encoding* crearía miles de columnas nuevas, haciendo que nuestro modelo sea pesado y lento.

El Target Encoding resuelve esto de una forma brillante: en lugar de crear columnas nuevas, reemplaza cada categoría con un número que representa el **promedio de la variable que queremos predecir** para ese grupo. Es como si le diéramos a cada etiqueta una pista directa sobre el resultado final.

### Un Escenario Real

Imagina a Sofía, una analista en una aplicación de entrega de comida. Ella tenía una columna con cinco mil tipos de platos diferentes y su modelo de tiempos de entrega fallaba por exceso de dimensiones. Al aplicar Target Encoding, convirtió esos nombres en el tiempo promedio de preparación de cada plato. Así, el modelo dejó de procesar texto inútil y empezó a entender que un sushi tarda más que una hamburguesa, logrando predicciones mucho más exactas y rápidas.

In [ ]:
# Instalación de la librería necesaria
try:
    import category_encoders as ce
except ImportError:
    import sys
    !{sys.executable} -m pip install category_encoders
    import category_encoders as ce

import pandas as pd
import numpy as np

# 1. Simulación de datos: Precios de alquiler por ciudad
data = {
    'Ciudad': ['Madrid', 'Madrid', 'Madrid', 'Barcelona', 'Barcelona', 'Sevilla', 'Sevilla', 'Bilbao'],
    'Precio': [1200, 1400, 1600, 1300, 1500, 900, 1100, 1000]
}

df = pd.DataFrame(data)
print('Dataset Original:')
print(df)
print()

## ¿Cómo funciona el Target Encoding?

Observen que para Madrid tenemos tres registros con precios de 1200, 1400 y 1600 euros. En lugar de dejar la palabra Madrid, el Target Encoding calcula el promedio (1400) y coloca ese valor exacto en cada fila donde aparece la ciudad.

Fíjense qué potente: ahora la columna **Ciudad** es numérica y contiene la esencia del impacto que tiene cada lugar en el precio. No solo estamos simplificando la estructura, sino que estamos inyectando conocimiento experto del negocio directamente en los datos.

In [ ]:
# 2. Aplicación de TargetEncoder sin suavizado para ver el concepto base
encoder_simple = ce.TargetEncoder(cols=['Ciudad'], smoothing=0)
df_encoded = encoder_simple.fit_transform(df['Ciudad'], df['Precio'])

df_resultado = df.copy()
df_resultado['Ciudad_Encoded'] = df_encoded['Ciudad']

print('Dataset transformado (Promedio directo):')
print(df_resultado)
print()

## El Peligro del Sobreajuste (Overfitting)

Si usamos el promedio directo, corremos el riesgo de que el modelo aprenda de memoria los datos de entrenamiento (Data Leakage). Para evitar esto, la librería **Category Encoders** implementa una técnica llamada **suavizado o smoothing**.

El suavizado mezcla el promedio de la categoría con el promedio global de todos los datos. Si una categoría aparece muy pocas veces (baja representatividad), confiamos más en el promedio global para no sesgar el modelo con un valor extremo.

### Ejemplo de Suavizado
Si el promedio global es 1250 y tenemos una ciudad con una sola vivienda de 2000, el encoder no le asignará 2000 directamente, sino un valor intermedio para ser cauteloso.

In [ ]:
# 3. Demostración de Suavizado (Smoothing)
# Añadimos una ciudad con un solo dato extremo
nuevo_dato = pd.DataFrame({'Ciudad': ['Valencia'], 'Precio': [2500]})
df_extremo = pd.concat([df, nuevo_dato], ignore_index=True)

promedio_global = df_extremo['Precio'].mean()
print('Promedio Global de todos los datos:', promedio_global)
print()

# Aplicamos TargetEncoder con un smoothing alto
encoder_smooth = ce.TargetEncoder(cols=['Ciudad'], smoothing=10.0)
df_smooth = encoder_smooth.fit_transform(df_extremo['Ciudad'], df_extremo['Precio'])

df_final = df_extremo.copy()
df_final['Ciudad_Smooth'] = df_smooth['Ciudad']

print('Resultado con Suavizado (Smoothing):')
print(df_final)
print()
print('Noten como Valencia no es 2500, sino que se acerca al promedio global debido al suavizado.')

## Conclusión y Beneficios

Dominar estas técnicas marca la diferencia entre un analista que solo sigue pasos y un profesional que sabe optimizar recursos.

*   **Eficiencia:** Reduce drásticamente la cantidad de columnas frente a One Hot Encoding.
*   **Inteligencia:** Crea una relación monótona con el objetivo, facilitando el trabajo de algoritmos de regresión y árboles.
*   **Rendimiento:** En datasets de seguros o retail con alta cardinalidad, se han observado mejoras de velocidad (de 30s a 2s por época) y de precisión (del 70% al 78%).

¡Muy bien! Has aprendido a reducir la complejidad sin sacrificar la información valiosa de tus variables categóricas. Sigue practicando con distintos valores de suavizado en tus proyectos para ver cómo evolucionan tus métricas.

¡Nos vemos en la siguiente lección para seguir transformando datos en conocimiento!